# Model Comparison and Analysis
Comparing XGBoost vs Random Forest performance on retail sales forecasting

## Importing libraries for loading, generating, and comparing models

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path
import sys

# Configuring graph properties

In [22]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Adding project root to path

In [23]:
project_root = Path.cwd().parent  # Going up one level from notebooks folder
sys.path.insert(0, str(project_root))

# Loading models, data, and feature columns

In [24]:
# Loading trained models
xgb_model = joblib.load('../trained_models/xgboost_model.pkl')
rf_model = joblib.load('../trained_models/randomforest_model.pkl')

# Loading test data
test_data = pd.read_csv('../data/processed/test_data.csv')

# Loading feature columns
with open('../trained_models/feature_columns.json', 'r') as f:
    feature_columns = json.load(f)

# Preparing data

In [25]:
X_test = test_data[feature_columns]
y_test = test_data['Units Sold']

print(f"Test set: {len(test_data):,} records")
print(f"Features: {len(feature_columns)}")

Test set: 9,390 records
Features: 37


# Generating predictions

In [26]:
xgb_predictions = xgb_model.predict(X_test)
rf_predictions = rf_model.predict(X_test)

# Creating the results dataframe

In [ ]:
results = test_data[['Date', 'Store ID', 'Units Sold']].copy()

# Reconstructing Category from the one-hot encoded columns
category_cols = [col for col in test_data.columns if col.startswith('Category_')]
results['Category'] = test_data[category_cols].idxmax(axis=1).str.replace('Category_', '')

results['XGBoost Pred'] = xgb_predictions
results['RF Pred'] = rf_predictions
results['XGBoost Error'] = results['Units Sold'] - results['XGBoost Pred']
results['RF Error'] = results['Units Sold'] - results['RF Pred']
results['XGBoost AbsError'] = np.abs(results['XGBoost Error'])
results['RF AbsError'] = np.abs(results['RF Error'])